# Transformações Power BI — Siplan RPS

#### Foi a base para o notebook nb_gold_siplan

Substitui as transformações do **Power Query** por Python puro (pandas/numpy), mantendo a lógica de negócio do RPS Sesc SP.

## Arquitetura

```
HIVE (siplan_data, siplan_solicitacao, siplan_acao, projetos, ...)
    ↓  Dataflow Gen2  ← os SQLs deste notebook viram queries aqui
LH_Siplan_Staging     ← tabelas raw (uma por fonte)
    ↓  Fabric Notebook ← este notebook, lendo com spark.table()
LH_Siplan_Prod        ← tabelas finais consumidas pelo Power BI
```

> **Desenvolvimento local:** as queries rodam via `pyodbc` (Kerberos/Hive). No Fabric, `query_to_df(sql)` é substituído por `spark.table('raw_xxx').toPandas()`.

## Tabelas staging e DataFrames (ordem de execução)

| # | DataFrame | Fonte Hive | Cardinalidade | Observações |
|---|---|---|---|---|
| 1a | `raw_acoes_df` | `siplan_acao` + `projetos` | 1 por atividade | Identificação, hierarquia, classificação, faixa etária, público |
| 1b | `raw_tags_df` | `siplan_tag` + `siplan_projeto_tag` | 1 por atividade × tag | `todas_as_tags` (pipe-separated) calculado em Python |
| 1c | `rps_parcial_df` | `raw_acoes_df` + `raw_tags_df` + `siplan_solicitacao` | 1 por atividade | `tag` (Avaliação STS), `tem_passagem`; alimenta seções 2, 3 e 4 |
| 2 | `datas_df` | `siplan_data` (agregado) | 1 por atividade | Datas, sessões, horas, flags de prazo (`30dias`, `90dias`, `60horas`), flags de ano, `autonomiaTemporal` |
| 3 | `contracts_df` | `siplan_solicitacao` + `siplan_acao` | 1 por solicitação | Contratos/passagem/hospedagem/filme; `autonomiaCusto`, métricas de custo e público |
| 4 | `autonomias_df` | `datas_df` + `contracts_df` + `rps_parcial_df` | 1 por atividade | Hierarquia DIREG (3) > STS (2) > UO (1); combina 4 fontes |
| 5 | `raw_projetos_df` | `siplan_acao` + `projetos` + `projeto_uo` | 1 por atividade | Metadados de projeto; `tem_pai` (bool) |
| 6 | `raw_datas_sessoes_df` | `siplan_data` (detalhe) | 1 por sessão | Local, horário, `TipoLocal`, `PeriodoDia`; **relatórios de detalhe** |
| 6b | `todas_as_datas_df` | `raw_datas_sessoes_df` (Python) | 1 por atividade | Formato "seg, dd/mm/yy HHhMM" por linha; entra na tabela base |
| 7 | `raw_acessibilidade_df` | `siplan_acessibilidade` | 1 por recurso | `tem_dispositivo` derivado via `.isin()` na tabela base |
| 8 | `raw_solicitacoes_df` | `siplan_solicitacao` | 1 por solicitação | Staging pura: todos os custos > 0; fonte para 8a, 9 e eventual refatoração da seção 3 |
| 8a | `solicitacoes_desc_df` | `raw_solicitacoes_df` | 1 por atividade | `item_desc` + `contratos/passagem/hospedagem_desc` + flags 0/1 `tem_*` |
| 9 | `raw_pcap_df` | `raw_solicitacoes_df` + `stg_pcap.vw_listagem_proposta` | 1 por solicitação | Vincula descricao PCAP (regex 13d) à proposta; `com_pcap` derivado via `.isin()` |
| 10 | `tabela_base_df` | todos os acima | **1 por atividade** | Join final; campos derivados: `faixa`, `periodicidade`, `tem_dispositivo`, `espaco_brincar`, `com_pcap` |

## Normalização de `item_de_custo` (seção 8a)

Categorias finais (19): `Contrato PJ` · `Contrato PF` · `Contrato Cooperativa` · `Hospedagem` · `Passagem Aérea` · `Turismo` · `Transporte` · `Exibição de Filmes` · `Ação esportiva e recreativa` · `Contratações diversas` · `Compras` · `Alimentação` · `Sonorização` · `Iluminação` · `Audiovisual` · `Locação` · `Comunicação` · `Acessibilidade` · `Outros`

Descartados: `Camarim` (tipos 1/2), `Água`, `Verificar`, `Estagiário`

## Periodicidade (seção 10)

Campo `periodicidade` calculado na tabela base com três regras:

| Regra | Condição | Resultado |
|---|---|---|
| Lista permanente | `subatividade` ∈ `SUBATIV_PERMANENTE` (12 subatividades) | `'permanente'` |
| Lista eventual | `subatividade` ∈ `SUBATIV_EVENTUAL` + Ações formativas/mediadas exceto Curso/Vivência | `'eventual'` |
| Curso / Vivência | `diascorridos > 90` **e** `qt_sessoes > 30` | `'permanente'` / `'eventual'` |
| Demais | subatividade não mapeada | `'s/i'` |

In [1]:
import warnings
import numpy as np
import pandas as pd
import pyodbc
from pathlib import Path

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok=True)
DATA_INICIAL = '2026-01-01'

# Ajuste a string abaixo se o DSN exigir parâmetros adicionais.
# Para autenticação Kerberos, normalmente basta usar o DSN configurado.
KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Authentication=Kerberos;'
# Se o driver exigir Trusted Connection no Windows, use:
# KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Trusted_Connection=Yes;'


def get_connection() -> pyodbc.Connection:
    return pyodbc.connect(KERBEROS_CONN_STR, autocommit=True)


def query_to_df(query: str) -> pd.DataFrame:
    with get_connection() as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*SQLAlchemy connectable.*',
                category=UserWarning,
                module='pandas.io.sql',
            )
            return pd.read_sql_query(query, conn)


MONTH_ABBR = {
    1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr', 5: 'mai', 6: 'jun',
    7: 'jul', 8: 'ago', 9: 'set', 10: 'out', 11: 'nov', 12: 'dez',
}

## 1. Ações

### 1a. Ações (siplan_acao) → `raw_acoes_df`

Base de ações programáticas com dados cadastrais, filtradas pelo período `DATA_INICIAL`.
Uma linha por `atividade_id`. Fonte: `stg_estatistico.siplan_acao` filtrada por `siplan_data`.

### 1b. Tags (siplan_tag + siplan_projeto_tag) → `raw_tags_df`

Tags vinculadas a cada atividade — diretamente ou herdadas do projeto.
`todas_as_tags` é calculada em Python após o load (`tag_nome` separados por `|`).
Fonte: `estatistico.siplan_acao` + `stg_estatistico.siplan_tag` + `siplan_projeto_tag`.

### 1c. RPS Parcial → `rps_parcial_df`

Tabela auxiliar com `servico`, `subatividade` e flag `tag = 'Avaliação STS'` por `atividade_id`.
Usada para calcular `autonomia` (seção 4) e `autonomiaCusto` (seção 3).

> `tem_passagem` é adicionado após a seção 8 (célula `rps-passagens-exec`).

In [2]:
# 1a. Query: ações programáticas do período
sql_raw_acoes = f'''
SELECT
    a.uo,
    a.status_atividade,
    a.atividade_id,
    a.nome,
    a.complemento,
    a.uso_interno,
    a.manutencao,
    a.regular,
    a.integracao_sgc,
    a.recomendacao_etaria,
    a.produtor,
    a.tem_parceria,
    a.contatofornecedores,
    a.areaprogramatica_nome                                            AS areaprog,
    a.desc_subprograma                                                 AS atividade,
    a.desc_modalidade                                                  AS subatividade,
    a.desc_realizacao                                                  AS servico,
    a.tipo_descricao                                                   AS tipo,
    a.subtipo_descricao                                                AS subtipo,
    a.formato,
    a.linguagem,
    a.idade_inicial,
    a.idade_final,
    a.estimativa_publico,
    a.lugares,
    a.precificacao_desc
FROM stg_estatistico.siplan_acao a
WHERE a.atividade_id IS NOT NULL
AND EXISTS (
    SELECT 1
    FROM stg_estatistico.siplan_data d
    WHERE d.atividade_id = a.atividade_id
      AND d.datainicio >= CAST(trunc(current_date, 'YYYY') AS timestamp)
)
'''

In [4]:
# 1a. Execução: raw_acoes_df
raw_acoes_df = query_to_df(sql_raw_acoes)
raw_acoes_df.columns    = [col.split('.')[-1] for col in raw_acoes_df.columns]
raw_acoes_df['atividade_id'] = raw_acoes_df['atividade_id'].astype(str).str.strip()
raw_acoes_df['servico']      = raw_acoes_df['servico'].astype(str).str.strip()
raw_acoes_df['subatividade'] = raw_acoes_df['subatividade'].astype(str).str.strip()

print(f'raw_acoes_df: {raw_acoes_df.shape}')

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_87992\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


raw_acoes_df: (33173, 26)


In [5]:
# 1b. Query: tags por atividade (diretas + herdadas do projeto)
# Usa f-string para {DATA_INICIAL} ser substituído corretamente.
# O filtro no WHERE limita às atividades do período — evita varredura total da tabela.
sql_raw_tags = f'''
SELECT
    acao.atividade_id                                                  AS atividade_id,
    siplan_tags.tag_nome                                               AS tag_nome,
    siplan_tags.tag_grupo                                              AS tag_grupo,
    siplan_tags.tag_origem                                             AS tag_origem
FROM estatistico.siplan_acao acao
LEFT JOIN (
    SELECT DISTINCT
        siplan_tag.atividade_id,
        siplan_tag.tag_nome,
        CASE WHEN siplan_tag.tag_grupo = 'Mobilização'
             THEN siplan_tag.tag_nome
             ELSE siplan_tag.tag_grupo
        END                                                            AS tag_grupo,
        CASE WHEN siplan_tag_na_acao.atividade_id IS NULL
             THEN 'acao'
             ELSE 'projeto'
        END                                                            AS tag_origem
    FROM (
        SELECT tag.atividade_id, tag.tag_nome, tag.grupo AS tag_grupo
        FROM stg_estatistico.siplan_tag tag
        UNION
        SELECT acao_proj_tag.atividade_id, proj_tag.tag_nome, proj_tag.tag_grupo
        FROM stg_estatistico.siplan_acao acao_proj_tag
        INNER JOIN (
            SELECT siplan_proj_tag.projeto_id, siplan_proj_tag.tag_nome,
                   siplan_proj_tag.grupo AS tag_grupo
            FROM stg_estatistico.siplan_projeto_tag siplan_proj_tag
            WHERE siplan_proj_tag.projeto_id IN (
                SELECT proj_tag.projeto_id
                FROM stg_estatistico.siplan_projeto_tag proj_tag
                GROUP BY proj_tag.projeto_id
                HAVING COUNT(*) <= 5
            )
        ) proj_tag ON acao_proj_tag.projeto_id = proj_tag.projeto_id
    ) siplan_tag
    LEFT JOIN (
        SELECT tag.atividade_id, tag.tag_nome, tag.grupo AS tag_grupo
        FROM stg_estatistico.siplan_tag tag
    ) siplan_tag_na_acao
        ON siplan_tag.atividade_id = siplan_tag_na_acao.atividade_id
       AND siplan_tag.tag_nome     = siplan_tag_na_acao.tag_nome
) siplan_tags ON acao.atividade_id = siplan_tags.atividade_id
WHERE siplan_tags.tag_nome IS NOT NULL
AND acao.atividade_id IS NOT NULL
AND EXISTS (
    SELECT 1
    FROM stg_estatistico.siplan_data d
    WHERE d.atividade_id = acao.atividade_id
      AND d.datainicio >= CAST(trunc(current_date, 'YYYY') AS timestamp)
)'''

In [6]:
# 1b. Execução: raw_tags_df + todas_as_tags (campo texto calculado em Python)
raw_tags_df = query_to_df(sql_raw_tags)
raw_tags_df.columns = [col.split('.')[-1] for col in raw_tags_df.columns]
raw_tags_df['atividade_id'] = raw_tags_df['atividade_id'].astype(str).str.strip()

todas_tags = (
    raw_tags_df.dropna(subset=['tag_nome'])
    .groupby('atividade_id')['tag_nome']
    .apply(lambda x: ' | '.join(sorted(x.unique())))
    .reset_index(name='todas_as_tags')
)
raw_tags_df = raw_tags_df.merge(todas_tags, on='atividade_id', how='left')

print(f'raw_tags_df: {raw_tags_df.shape}')
print(f'atividades com tags: {raw_tags_df["atividade_id"].nunique()}')

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_87992\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


raw_tags_df: (35988, 5)
atividades com tags: 19382


In [37]:
# 1c. rps_parcial_df — base RPS: serviço, subatividade, tag STS
# tem_passagem será adicionado após a seção 8 (ver rps-passagens-exec)
sts_df = (
    raw_tags_df[raw_tags_df['tag_nome'] == 'Avaliação STS'][['atividade_id']]
    .assign(tag='Avaliação STS')
    .drop_duplicates()
)

rps_parcial_df = (
    raw_acoes_df[['atividade_id', 'servico', 'subatividade']]
    .merge(sts_df[['atividade_id', 'tag']], on='atividade_id', how='left')
    .assign(tag=lambda d: d['tag'].fillna(''))
    .drop_duplicates(subset=['atividade_id'])
)

print(f'rps_parcial_df: {rps_parcial_df.shape}  ← tem_passagem adicionado na seção 8')

rps_parcial_df: (32831, 4)  ← tem_passagem adicionado na seção 8


### 1d. Precificação → `precif_df`

Parsing de `precificacao_desc` para derivar `gratuito`, `maior_valor` e `menor_valor` por `atividade_id`.
Execute a célula de exploração primeiro para validar os padrões de texto antes de ajustar o regex.

In [ ]:
# 1d. Parser de precificacao_desc → gratuito, maior_valor, menor_valor
#
# Regras:
#   - Extrai todos os valores "R$ XX.YY" ou "R$ XX,YY" via regex
#   - Detecta "grátis" / "gratuito" / "aberto" no texto (case-insensitive)
#   - gratuito = 'Sim' se não há valor pago (>0); 'Não' caso contrário
#   - maior_valor = max dos valores pagos encontrados (None se tudo grátis ou sem info)
#   - menor_valor = 0 se "grátis/aberto" coexiste com valores pagos; min dos pagos caso contrário

import re

# Ponto ou vírgula como separador decimal: "R$ 10.00" ou "R$ 10,00"
_RE_VALOR  = re.compile(r'R\$\s*(\d+[.,]\d{2})', re.IGNORECASE)
# "grátis", "gratis", "gratuito" ou "aberto" → acesso livre
_RE_GRATIS = re.compile(r'gr[áa]tis|gratuito|aberto', re.IGNORECASE)


def _str_to_float(s: str) -> float:
    # Normaliza separador decimal: "10,00" → "10.00", "10.00" → "10.00"
    return float(s.replace(',', '.'))


def parse_precificacao(desc) -> tuple:
    """Retorna (gratuito, maior_valor, menor_valor) a partir de precificacao_desc."""
    if pd.isna(desc) or str(desc).strip() == '':
        return ('Não', None, None)

    s = str(desc)
    tem_gratis = bool(_RE_GRATIS.search(s))

    raw_vals = _RE_VALOR.findall(s)
    valores = []
    for v in raw_vals:
        try:
            valores.append(_str_to_float(v))
        except ValueError:
            pass

    pagos = [v for v in valores if v > 0]

    if not pagos and not tem_gratis:
        return ('Não', None, None)

    if not pagos:
        # Apenas grátis / aberto
        return ('Sim', 0.0, 0.0)

    # Há valores pagos
    maior = max(pagos)
    # "grátis/aberto" coexiste com pago → opção gratuita existe → menor = 0
    menor = 0.0 if tem_gratis else min(pagos)
    return ('Não', maior, menor)


# ── Aplica e exibe diagnóstico ────────────────────────────────────────────────
_res = raw_acoes_df['precificacao_desc'].apply(parse_precificacao)
precif_df = pd.DataFrame(_res.tolist(), columns=['gratuito', 'maior_valor', 'menor_valor'],
                         index=raw_acoes_df.index)
precif_df.insert(0, 'atividade_id', raw_acoes_df['atividade_id'])

print("gratuito:")
print(precif_df['gratuito'].value_counts())
print()
print("maior_valor — estatísticas:")
print(precif_df['maior_valor'].describe())
print()
print("menor_valor — estatísticas:")
print(precif_df['menor_valor'].describe())
print()

# Amostra de casos mistos (grátis/aberto + pago na mesma atividade)
_mistos = precif_df[(precif_df['gratuito'] == 'Não') & (precif_df['menor_valor'] == 0)]
print(f"Casos mistos (pago + grátis/aberto): {len(_mistos)}")
if len(_mistos) > 0:
    _check = raw_acoes_df.loc[_mistos.index, ['atividade_id', 'precificacao_desc']].head(5)
    for _, row in _check.iterrows():
        print(f"  {row['atividade_id']}: {str(row['precificacao_desc'])[:90]}")

## 2. Datas (siplan_data)

Tabela de referência de datas por `atividade_id`, construída diretamente de `siplan_data` — **sem depender da view** `vw_siplan_datas_totais`.

Consultada **uma única vez** aqui. Toda a lógica da view foi trazida para este SQL.

**Campos produzidos:**
- `primeiradata`, `ultimadata` — primeira e última sessão da atividade
- `qt_sessoes` — número de sessões únicas (`sessao_id` distintos)
- `qt_datas_distintas` — número de datas distintas com sessão (dias de calendário em que há pelo menos uma sessão)
- `qt_horas` — total de horas (soma de `datafinal − datainicio` de cada sessão)
- `tempo_sessao` — duração média por sessão (`qt_horas / qt_sessoes`)
- `diascorridos` — dias corridos entre primeira e última sessão (`DATEDIFF`)
- `` `30dias` `` — 1 se `diascorridos > 30`
- `` `90dias` `` — 1 se `diascorridos > 90`
- `` `60horas` `` — 1 se `qt_horas > 60`
- `ano_2018` … `ano_2026` — flag `1/0` por ano (2020 e 2021 incluídos; exclusão feita nos relatórios)

> Lakehouse Staging: **`raw_datas`**

In [ ]:
# Lakehouse Staging: raw_datas
sql_datas = '''
SELECT
    atividade_id,
    MIN(datainicio)                                                                AS primeiradata,
    MAX(datainicio)                                                                AS ultimadata,
    COUNT(DISTINCT sessao_id)                                                      AS qt_sessoes,
    COUNT(DISTINCT TO_DATE(datainicio))                                            AS qt_datas_distintas,
    SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)           AS qt_horas,
    SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)
        / COUNT(DISTINCT sessao_id)                                                AS tempo_sessao,
    DATEDIFF(MAX(datainicio), MIN(datainicio))                                      AS diascorridos,
    CASE WHEN DATEDIFF(MAX(datainicio), MIN(datainicio)) > 30  THEN 1 ELSE 0 END   AS `30dias`,
    CASE WHEN DATEDIFF(MAX(datainicio), MIN(datainicio)) > 90  THEN 1 ELSE 0 END   AS `90dias`,
    CASE WHEN SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)
              > 60 THEN 1 ELSE 0 END                                               AS `60horas`,
    MAX(CASE WHEN YEAR(datainicio) = 2018 THEN 1 ELSE 0 END)                       AS ano_2018,
    MAX(CASE WHEN YEAR(datainicio) = 2019 THEN 1 ELSE 0 END)                       AS ano_2019,
    MAX(CASE WHEN YEAR(datainicio) = 2020 THEN 1 ELSE 0 END)                       AS ano_2020,
    MAX(CASE WHEN YEAR(datainicio) = 2021 THEN 1 ELSE 0 END)                       AS ano_2021,
    MAX(CASE WHEN YEAR(datainicio) = 2022 THEN 1 ELSE 0 END)                       AS ano_2022,
    MAX(CASE WHEN YEAR(datainicio) = 2023 THEN 1 ELSE 0 END)                       AS ano_2023,
    MAX(CASE WHEN YEAR(datainicio) = 2024 THEN 1 ELSE 0 END)                       AS ano_2024,
    MAX(CASE WHEN YEAR(datainicio) = 2025 THEN 1 ELSE 0 END)                       AS ano_2025,
    MAX(CASE WHEN YEAR(datainicio) = 2026 THEN 1 ELSE 0 END)                       AS ano_2026
FROM stg_estatistico.siplan_data
GROUP BY atividade_id
'''

In [39]:
def transform_datas(df: pd.DataFrame, rps_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- data/hora da primeira sessão ---
    df['primeiradata'] = pd.to_datetime(df['primeiradata'], errors='coerce')
    df['PrimeiraData']     = df['primeiradata'].dt.date
    df['PrimeiraHora']     = df['primeiradata'].dt.time
    df['PrimeiraDataHora'] = df['primeiradata'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['mes'] = df['primeiradata'].dt.month.map(MONTH_ABBR)

    # --- tempo médio da sessão (truncado a 1 decimal) ---
    df['tempo_da_sessao'] = (
        np.floor(pd.to_numeric(df['tempo_sessao'], errors='coerce').fillna(0) * 10) / 10
    )

    # --- flags de prazo (binário → 'sim'/'0') ---
    # 30dias: nº de datas distintas com sessão > 30  (≠ diascorridos > 30)
    # 90dias: diascorridos > 90
    # 60horas: total de horas > 60
    # Todos já calculados no SQL; apenas mapeamos para 'sim'/'0'
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    for col in ['30dias', '90dias', '60horas']:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(bool_map).fillna('0')

    df['ExtrapolaDataHora'] = np.where(
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'), 'sim', '0'
    )

    # --- autonomia temporal bruta ---
    # DIREG: ultrapassa 90 dias corridos OU 60 horas totais
    # STS  : mais de 30 datas distintas com sessão
    # UO   : demais casos
    conditions = [
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'),
        df['30dias'] == 'sim',
    ]
    df['autonomiaTemporal'] = np.select(conditions, ['DIREG', 'STS'], default='UO')

    # --- ajuste por serviço (via RPS Parcial) ---
    # A autonomia temporal só faz sentido para serviços com acúmulo de carga/dias.
    # Para os demais, UO é sempre o nível correto independente das datas.
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    usa_temporal = (
        df['servico'].isin({'Curso', 'Ioga'}) |
        df['servico'].str.startswith('Desenvolvimento') |
        df['subatividade'].str.startswith('Ações')
    )
    df['autonomiaTemporal'] = np.where(usa_temporal, df['autonomiaTemporal'], 'UO')

    # servico/subatividade já estarão na tabela base; remove daqui para evitar duplicatas
    df = df.drop(columns=['servico', 'subatividade'])

    return df


datas_df = transform_datas(query_to_df(sql_datas), rps_parcial_df)
datas_df.shape

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_80416\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


(449635, 27)

## 3. Projetos (siplan_acao + projetos + projeto_uo)

Metadados de projeto por `atividade_id`. Cada linha = uma atividade (deduplicada).

**Campos produzidos:**
- `projeto_id`, `projeto_nome`, `projeto_complemento` — identificação do projeto
- `institucional`, `projeto_categoria`, `tag_projeto`, `tag_grupo_projeto` — classificação
- `projeto_uo_nome`, `projeto_descricao`, `projeto_comunicacao`, `projeto_conceitual` — dados da UO
- `tem_pai` (bool) — indica se o projeto tem pai hierárquico em `projeto_uo`

**Decisões de design:**
- Join via `projeto_uo.id` cobre 100% dos projetos; 409 atividades com `projeto_id` órfão preservadas via LEFT JOIN
- JOIN com tabela pai removido — `tem_pai` é suficiente para derivar flags como `espaco_brincar`
- Filtro pós-load para manter apenas atividades do período corrente (`raw_acoes_df`)

> Lakehouse Staging: **`raw_projetos`**


In [ ]:
# Lakehouse Staging: raw_projetos
sql_raw_projetos = f'''
SELECT
    a.atividade_id,

    -- Metadados do projeto (de projetos)
    p.projeto_id,
    p.projeto                       AS projeto_nome,
    p.projeto_complemento           AS projeto_complemento,
    p.institucional                 AS institucional,
    p.categoria_projeto             AS projeto_categoria,
    p.tag_projeto                   AS tag_projeto,
    p.tag_grupo_projeto             AS tag_grupo_projeto,

    -- Projeto UO — nível próprio
    pu.nome                         AS projeto_uo_nome,
    pu.descricao                    AS projeto_descricao,
    pu.descricao_comunicacao        AS projeto_comunicacao,
    pu.descricao_conceitual         AS projeto_conceitual,
    CASE WHEN pu.projetopai_id IS NOT NULL THEN TRUE ELSE FALSE END AS tem_pai

FROM stg_estatistico.siplan_acao a
LEFT JOIN stg_estatistico.projetos p
       ON a.projeto_id = p.projeto_id
LEFT JOIN stg_estatistico.projeto_uo pu
       ON p.projeto_id = pu.id
WHERE a.projeto_id IS NOT NULL
  AND a.atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''

In [ ]:
raw_projetos_df = query_to_df(sql_raw_projetos)
raw_projetos_df.columns = [col.split(".")[-1] for col in raw_projetos_df.columns]
raw_projetos_df["atividade_id"] = raw_projetos_df["atividade_id"].astype(str).str.strip()

# Filtra apenas atividades do período corrente (já carregadas em raw_acoes_df)
raw_projetos_df = raw_projetos_df[
    raw_projetos_df["atividade_id"].isin(raw_acoes_df["atividade_id"])
]

# siplan_acao pode ter múltiplas linhas por atividade_id;
# os dados de projeto são iguais em todas — mantém só a primeira
raw_projetos_df = raw_projetos_df.drop_duplicates(subset=["atividade_id"])

# Normaliza coluna de nome
for col in ["projeto_uo_nome"]:
    if col in raw_projetos_df.columns:
        raw_projetos_df[col] = raw_projetos_df[col].fillna("").astype(str).str.strip()

print(f"raw_projetos_df: {raw_projetos_df.shape}")
print(f"atividades com projeto: {raw_projetos_df['atividade_id'].nunique()}")
print(f"com pai: {raw_projetos_df['tem_pai'].sum()}")

## 4. Datas / Sessões (siplan_data)

Sessões de atividades com local, horário, tipologia e derivações temporais.
Substitui a query `raw_datas_sessoes` do Power Query. Cada linha = uma sessão.

O JOIN com subquery traz `primeira_data` e `primeira_hora` (data e hora da primeira sessão de cada atividade).

**Campos produzidos:**
- `atividade_id`, `sessao_id`, `uo` — identificação
- `datainicio`, `Data` (datetime meia-noite), `diferenca` (horas inteiras), `Duracao` (timedelta) — tempo
- `HoraCheia` (hora cheia), `horaCerta` (hora exata), `horaCertaTxt` (texto HH:MM:SS) — horário
- `PeriodoDia` — Madrugada (0–4h) / de manhã (5–11h) / à tarde (12–17h) / à noite (18–23h)
- `local`, `TipologiaLocal` (ex-`grupo_nome`), `TipoLocal` (`online`/`externa`/`na UO`) — local
- `ano`, `Mes`, `mesTxt`, `dia`, `diaSemama`, `diaSemanaTxt`, `Semana do Ano` — partes de data
- `primeira_data`, `primeira_hora`, `mes1o`, `mes1oTxt` — mês de referência da atividade

**Uso:** relatórios de detalhe por sessão. **Não entra no join da tabela base.**

**Derivado em seguida:** `todas_as_datas_df` — agrega datas de cada atividade em texto `"seg, dd/mm/yy HHhMM"` (uma linha por data única, ordem cronológica). Entra na tabela base.

> Lakehouse Staging: **`raw_datas_sessoes`**

In [21]:
# Lakehouse Staging: raw_datas_sessoes
sql_raw_datas_sessoes = f'''
SELECT
    d.sessao_id,
    d.atividade_id,
    d.datainicio,
    d.datafinal,
    d.local_id,
    d.local_nome,
    d.grupo_id,
    d.grupo_nome,
    d.geac,
    d.correcao_local,
    d.uo_local,
    TO_DATE(m.primeira_ts)                  AS primeira_data,
    date_format(m.primeira_ts, 'HH:mm:ss')  AS primeira_hora
FROM stg_estatistico.siplan_data AS d
JOIN (
    SELECT
        atividade_id,
        MIN(datainicio) AS primeira_ts
    FROM stg_estatistico.siplan_data
    GROUP BY atividade_id
) AS m ON m.atividade_id = d.atividade_id
WHERE TO_DATE(d.datainicio) >= DATE '{DATA_INICIAL}'
'''


In [22]:
# Nota: diaSemama preserva o typo original do Power Query (era "semana").
# Renomear quebraria relatórios existentes que referenciem essa coluna.
WEEKDAY_ABBR = {0: "seg", 1: "ter", 2: "qua", 3: "qui",
                4: "sex", 5: "sab", 6: "dom"}


def transform_datas_sessoes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.split(".")[-1] for col in df.columns]

    # Tipos base
    df["atividade_id"]  = df["atividade_id"].astype(str).str.strip()
    df["sessao_id"]     = df["sessao_id"].astype(str).str.strip()
    df["datainicio"]    = pd.to_datetime(df["datainicio"],    errors="coerce")
    df["datafinal"]     = pd.to_datetime(df["datafinal"],     errors="coerce")
    df["primeira_data"] = pd.to_datetime(df["primeira_data"], errors="coerce")

    # Remove sessões com datas fora do intervalo datetime64[ns] (ex: anos > 2262)
    n_before = len(df)
    df = df[df["datainicio"].notna()].copy()
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"[AVISO] {n_dropped} sessão(ões) descartada(s): datainicio inválida ou fora dos limites")

    # Exclui grupo "37" (tipologia sem validade)
    df = df[df["grupo_nome"] != "37"].copy()

    # Duração e diferença em horas inteiras
    df["Duracao"]   = df["datafinal"] - df["datainicio"]
    df["diferenca"] = (df["Duracao"].dt.total_seconds() / 3600).astype(int)

    # Colunas de tempo derivadas de datainicio
    dt = df["datainicio"].dt
    df["Data"]         = dt.normalize()                        # datetime na meia-noite
    df["HoraCheia"]    = df["datainicio"].dt.floor("h").dt.time  # hora cheia (sem minutos)
    df["horaCerta"]    = dt.time                               # hora exata
    df["horaCertaTxt"] = dt.strftime("%H:%M:%S")

    # PeriodoDia baseado na hora cheia
    hora = dt.hour
    df["PeriodoDia"] = np.select(
        [hora < 5, hora < 12, hora < 18],
        ["Madrugada", "de manhã", "à tarde"],
        default="à noite"
    )

    # TipoLocal — vetorizado (máscara de maior prioridade aplicada por último)
    g = df["grupo_nome"].fillna("")
    l = df["local_nome"].fillna("")
    df["TipoLocal"] = "na UO"
    df.loc[g == "Fora da Unidade",              "TipoLocal"] = "externa"
    df.loc[l.str.contains("Online",       na=False), "TipoLocal"] = "online"
    df.loc[l.str.contains("Sesc Digital", na=False), "TipoLocal"] = "online"
    df.loc[g.str.contains("Online",       na=False), "TipoLocal"] = "online"

    # Renomeia e formata texto
    df = df.rename(columns={"grupo_nome": "TipologiaLocal", "local_nome": "local"})
    df["local"] = df["local"].str.title()

    # uo = primeiros 2 dígitos do atividade_id
    df["uo"] = df["atividade_id"].str[:2].astype(int)

    # Partes de data (de datainicio)
    df["ano"]           = dt.year
    df["Mes"]           = dt.month
    df["mesTxt"]        = dt.month.map(MONTH_ABBR)
    df["dia"]           = dt.day
    df["diaSemama"]     = dt.dayofweek             # seg=0, dom=6
    df["diaSemanaTxt"]  = dt.dayofweek.map(WEEKDAY_ABBR)
    df["Semana do Ano"] = dt.isocalendar().week.astype(int)

    # Partes de data (de primeira_data — mês da 1ª sessão da atividade)
    p = df["primeira_data"].dt
    df["mes1o"]    = p.month
    df["mes1oTxt"] = p.month.map(MONTH_ABBR)

    # Remove colunas não consumidas downstream
    df = df.drop(columns=["local_id", "grupo_id", "correcao_local",
                          "uo_local", "geac", "datafinal"], errors="ignore")

    col_order = [
        "atividade_id", "sessao_id", "uo",
        "datainicio", "Data", "diferenca", "Duracao",
        "HoraCheia", "horaCerta", "horaCertaTxt", "PeriodoDia",
        "local", "TipologiaLocal", "TipoLocal",
        "ano", "Mes", "mesTxt", "dia", "diaSemama", "diaSemanaTxt", "Semana do Ano",
        "primeira_data", "primeira_hora", "mes1o", "mes1oTxt",
    ]
    return df[[c for c in col_order if c in df.columns]]


In [23]:
raw_datas_sessoes_df = query_to_df(sql_raw_datas_sessoes)
raw_datas_sessoes_df = transform_datas_sessoes(raw_datas_sessoes_df)

print(f'raw_datas_sessoes_df: {raw_datas_sessoes_df.shape}')
print(f'atividades únicas:     {raw_datas_sessoes_df["atividade_id"].nunique()}')
print(f'\nTipoLocal:')
print(raw_datas_sessoes_df['TipoLocal'].value_counts())


C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_80416\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


[AVISO] 8 sessão(ões) descartada(s): datainicio inválida ou fora dos limites
raw_datas_sessoes_df: (484686, 25)
atividades únicas:     32826

TipoLocal:
TipoLocal
na UO      475251
externa      8546
online        889
Name: count, dtype: int64


In [24]:
# Todas as datas por atividade — formato "seg, dd/mm/yy HHhMM"
# Deduplica por (atividade_id, datainicio) antes de agregar
_d = (
    raw_datas_sessoes_df[['atividade_id', 'datainicio', 'diaSemanaTxt']]
    .drop_duplicates()
    .sort_values(['atividade_id', 'datainicio'])
    .assign(_linha=lambda d:
        d['diaSemanaTxt'] + ', ' + d['datainicio'].dt.strftime('%d/%m/%y %Hh%M')
    )
)

todas_as_datas_df = (
    _d.groupby('atividade_id')['_linha']
    .apply('\n'.join)
    .reset_index(name='todas_as_datas')
)

print(f'todas_as_datas_df: {todas_as_datas_df.shape}')
print()
print('Exemplo:')
print(todas_as_datas_df['todas_as_datas'].iloc[0])

todas_as_datas_df: (32826, 2)

Exemplo:
seg, 01/11/02 10h45


## 5. Acessibilidade (siplan_acessibilidade)

Recursos de acessibilidade vinculados a cada `atividade_id`.
Usada em dois contextos:
- **Relatórios** — detalha quais recursos estão associados a cada atividade (uma linha por recurso)
- **Tabela base** — flag `tem_dispositivo` (0/1): calculada via `.isin()` na seção 10

**Campos:**
- `atividade_id`, `acessibilidade_id`, `identificacao` — direto do banco
- `tem_dispositivo` — sempre `'sim'` (filtro `acessibilidade_id <> 9` já exclui os sem dispositivo)
- `uo` — primeiros 2 dígitos do `atividade_id`

In [25]:
# Lakehouse Staging: raw_acessibilidade
sql_raw_acessibilidade = f'''
SELECT
    atividade_id,
    acessibilidade_id,
    identificacao,
    CASE
        WHEN acessibilidade_id = 9 THEN 'não' ELSE 'sim'
    END AS tem_dispositivo
FROM stg_estatistico.siplan_acessibilidade
WHERE acessibilidade_id <> 9
  AND acessibilidade_id IS NOT NULL
  AND atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''


In [26]:
raw_acessibilidade_df = query_to_df(sql_raw_acessibilidade)
raw_acessibilidade_df.columns = [col.split('.')[-1] for col in raw_acessibilidade_df.columns]
raw_acessibilidade_df['atividade_id'] = raw_acessibilidade_df['atividade_id'].astype(str).str.strip()
raw_acessibilidade_df['uo']           = raw_acessibilidade_df['atividade_id'].str[:2].astype(int)

# Filtra apenas atividades do período corrente
raw_acessibilidade_df = raw_acessibilidade_df[
    raw_acessibilidade_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()

print(f'raw_acessibilidade_df: {raw_acessibilidade_df.shape}')
print(f'atividades únicas:     {raw_acessibilidade_df["atividade_id"].nunique()}')
print()
print('recursos:')
print(raw_acessibilidade_df['identificacao'].value_counts())


C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_80416\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


raw_acessibilidade_df: (1558, 5)
atividades únicas:     1232

recursos:
identificacao
Tradução em Libras    1069
Audiodescrição         171
Recursos Táteis        143
Braille                128
Texto Ampliado          28
Audioguia               18
Closed Caption           1
Name: count, dtype: int64


## 6. Solicitações (siplan_solicitacao)

Staging pura de todas as solicitações com `custo > 0` para o período corrente.
Cada linha = uma solicitação; várias por `atividade_id`.

**Campos:** `atividade_id`, `solicitacao_id`, `custo`, `area`, `item_grupo`, `nome_item`, `descricao`, `status_solic`, `obs_solicitacao`

**Consumidores:**
- Seção 8a → `solicitacoes_desc_df` (descrições e flags de custo por atividade)
- Seção 9 → `raw_pcap_df` (filtra Administrativo + regex PCAP em `descricao`)
- Seção 3 → `contracts_df` via `build_contracts()` (célula `contracts-exec`, após esta seção)
- Seção 1c → `tem_passagem` em `rps_parcial_df` (célula `rps-passagens-exec`, após esta seção)

> Lakehouse Staging: **`raw_solicitacoes`**

In [27]:
# Lakehouse Staging: raw_solicitacoes
sql_raw_solicitacoes = f'''
SELECT
    solic.atividade_id,
    NVL(solic.solicitacao_id, 0)        AS solicitacao_id,
    NVL(solic.custo, 0)                 AS custo,
    solic.solicitacao_area              AS area,
    solic.item_grupo                    AS item_grupo,
    solic.nome_item                     AS nome_item,
    solic.descricao                     AS descricao,
    solic.status_solicitacao            AS status_solic,
    solic.obs_solicitacao               AS obs_solicitacao
FROM stg_estatistico.siplan_solicitacao solic
WHERE solic.datainicio > '{DATA_INICIAL}'
  AND NVL(solic.custo, 0) > 0
  AND solic.atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''


In [28]:
raw_solicitacoes_df = query_to_df(sql_raw_solicitacoes)
raw_solicitacoes_df.columns = [col.split('.')[-1] for col in raw_solicitacoes_df.columns]
raw_solicitacoes_df['atividade_id']   = raw_solicitacoes_df['atividade_id'].astype(str).str.strip()
raw_solicitacoes_df['solicitacao_id'] = raw_solicitacoes_df['solicitacao_id'].astype(str).str.strip()
raw_solicitacoes_df['custo']          = pd.to_numeric(raw_solicitacoes_df['custo'], errors='coerce').fillna(0)

# Filtra apenas atividades do período corrente
raw_solicitacoes_df = raw_solicitacoes_df[
    raw_solicitacoes_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()

print(f'raw_solicitacoes_df: {raw_solicitacoes_df.shape}')
print(f'atividades unicas:   {raw_solicitacoes_df["atividade_id"].nunique()}')
print()
print('areas:')
print(raw_solicitacoes_df['area'].value_counts())


C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_80416\4155705979.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


raw_solicitacoes_df: (46285, 9)
atividades unicas:   17168

areas:
area
Alimentação             24958
Administrativo          20978
Serviços Gerais           266
Comunicação                38
Operação de Montagem       30
Infraestrutura             15
Name: count, dtype: int64


In [42]:
# ── tem_passagem: derivado de raw_solicitacoes_df (elimina sql_passagens) ─
# Equivalente a WHERE item_grupo LIKE '%Passagem%'; filtra custo > 0
# (passagens com custo = 0 não devem gerar autonomia STS)
_passagens_ids = raw_solicitacoes_df[
    raw_solicitacoes_df['item_grupo'].str.contains('Passagem', case=False, na=False)
]['atividade_id']

rps_parcial_df['tem_passagem'] = np.where(
    rps_parcial_df['atividade_id'].isin(_passagens_ids), 'Sim', 'Não'
)
print(f'tem_passagem=Sim: {(rps_parcial_df["tem_passagem"] == "Sim").sum()}')

tem_passagem=Sim: 530


## 8b. Contratos (siplan_solicitacao)

Solicitações administrativas de contratação por `atividade_id`, com custos e métricas calculadas.

**Construído em Python** sobre `raw_solicitacoes_df` (seção 8) — elimina a query Hive original com 5 subqueries.
Filtra `area = 'Administrativo'` com `item_grupo`/`nome_item` contendo: **contrato, passagem, hospedagem ou filme**.

**Campos calculados em Python:**
- `custo_contratos_total`, `n_contratos` — soma/contagem de contrato+filme por `atividade_id` (groupby)
- `custo_total`, `n_solic` — soma/contagem de **todas** as solicitações (groupby)
- `publico_sessao`, `tipo_per_capita` — de `raw_acoes_df` (substituem JOIN com `siplan_acao` do Hive)
- `sessoes`, `horas` — merge com `datas_df`
- `acima15mil`, `acima20mil`, `acima100mil` — flags de custo individual
- `publico` — `tipo_per_capita=0` → `sessoes × publico_sessao`; caso contrário → `publico_sessao` direto
- `por_sessao`, `por_hora`, `per_capita` — sobre `custo_contratos_total`
- `autonomiaCusto` — DIREG (>100k), STS (>20k com limiar), STS-20 (>20k), STS-15 (>15k), UO
- `por_hora_valido` — `por_hora` apenas para serviços em `SERVICOS_POR_HORA`

> **Execução:** diferida para após a seção 8 (célula `contracts-exec`) — depende de `raw_solicitacoes_df`.

In [31]:
SERVICOS_LIMIAR_20K  = {'Debate', 'Seminário', 'Visita Mediada'}
SUBATIV_LIMIAR_20K   = {'Ações formativas', 'Ações mediadas', 'Passeios', 'Viagens'}
SERVICOS_POR_HORA    = {'Curso', 'Oficina', 'Vivência', 'Seminário', 'Mediação', 'Visita Mediada', 'Intervenção urbana'}
SUBATIV_POR_HORA     = {'Multipráticas recreativas', 'Passeios', 'Viagens', 'Colônias recreativas'}


def build_contracts(
    raw_solicitacoes_df: pd.DataFrame,
    raw_acoes_df: pd.DataFrame,
    datas_df: pd.DataFrame,
    rps_parcial_df: pd.DataFrame,
) -> pd.DataFrame:
    """Constrói contracts_df a partir de raw_solicitacoes_df + raw_acoes_df.

    Substitui sql_contratos (5 subqueries Hive) por groupby Python sobre
    raw_solicitacoes_df, que já está disponível na seção 8.
    Saída: uma linha por solicitação; métricas agregadas repetidas por atividade_id.
    """
    solic = raw_solicitacoes_df.copy()

    # --- classificação de tipo por item_grupo / nome_item ---
    grupo_lower = solic['item_grupo'].fillna('').str.lower()
    item_lower  = solic['nome_item'].fillna('').str.lower()

    is_admin      = solic['area'].str.strip() == 'Administrativo'
    is_contrato   = grupo_lower.str.contains('contrato') | item_lower.str.contains('contrato')
    is_filme      = grupo_lower.str.contains('filme')
    is_passagem   = grupo_lower.str.contains('passagem') | item_lower.str.contains('passagem')
    is_hospedagem = grupo_lower.str.contains('hospedagem') | item_lower.str.contains('hospedagem')

    # Filtro principal: admin + (contrato | passagem | hospedagem | filme)
    # Espelha o WHERE do FROM principal da query Hive original
    mask_main = is_admin & (is_contrato | is_filme | is_passagem | is_hospedagem)
    df = solic[mask_main].copy()

    # Renomeia para manter compatibilidade com o código consumidor (build_autonomias, base)
    df = df.rename(columns={'item_grupo': 'grupo', 'nome_item': 'item', 'descricao': 'complemento'})
    # Remove sufixo "[...]" que a view do Hive às vezes insere no grupo
    df['grupo'] = df['grupo'].astype(str).str.split('[').str[0].str.strip()

    # --- agregados por atividade_id (substituem os 4 subqueries do Hive) ---

    # custo_contratos_total / n_contratos: apenas contrato+filme, área Administrativo
    # Semântica original: base das métricas per-capita (por_sessao, por_hora, per_capita)
    mask_cf = is_admin & (is_contrato | is_filme)
    agg_cf = (
        solic[mask_cf]
        .groupby('atividade_id', as_index=False)
        .agg(custo_contratos_total=('custo', 'sum'), n_contratos=('solicitacao_id', 'count'))
    )

    # custo_total / n_solic: todas as solicitações (raw_solicitacoes_df já filtrou custo > 0)
    agg_total = (
        solic
        .groupby('atividade_id', as_index=False)
        .agg(custo_total=('custo', 'sum'), n_solic=('solicitacao_id', 'count'))
    )

    df = df.merge(agg_cf,    on='atividade_id', how='left')
    df = df.merge(agg_total, on='atividade_id', how='left')
    df[['custo_contratos_total', 'custo_total']] = (
        df[['custo_contratos_total', 'custo_total']].fillna(0)
    )
    df['n_contratos'] = df['n_contratos'].fillna(0).astype(int)
    df['n_solic']     = df['n_solic'].fillna(0).astype(int)

    # --- campos de público (substituem o INNER JOIN com siplan_acao da query Hive) ---
    acoes_pub = raw_acoes_df[['atividade_id', 'lugares', 'estimativa_publico', 'servico']].copy()

    # publico_sessao: lugares preferido, fallback estimativa_publico, mínimo 1
    # Espelha: NVL(lugares, NVL(estimativa_publico, 1)) com CASE quando <= 0 → 1
    pub_raw = acoes_pub['lugares'].where(acoes_pub['lugares'].notna(), acoes_pub['estimativa_publico'])
    acoes_pub['publico_sessao'] = pd.to_numeric(pub_raw, errors='coerce').fillna(1).clip(lower=1).astype(int)
    acoes_pub['capacidade']     = pd.to_numeric(acoes_pub['lugares'],            errors='coerce')
    acoes_pub['estimativa']     = pd.to_numeric(acoes_pub['estimativa_publico'], errors='coerce')

    # tipo_per_capita=1 → público é total (não se multiplica por sessões)
    # Espelha: CASE WHEN LOWER(desc_realizacao) IN ('curso', 'seminário') THEN 1 ELSE 0
    acoes_pub['tipo_per_capita'] = acoes_pub['servico'].isin(['Curso', 'Seminário']).astype(int)

    df = df.merge(
        acoes_pub[['atividade_id', 'publico_sessao', 'capacidade', 'estimativa', 'tipo_per_capita']],
        on='atividade_id', how='left',
    )
    df['publico_sessao']  = pd.to_numeric(df['publico_sessao'],  errors='coerce').fillna(1).clip(lower=1).astype(int)
    df['tipo_per_capita'] = pd.to_numeric(df['tipo_per_capita'], errors='coerce').fillna(0)

    # --- sessoes e horas reais (de datas_df, gerado na seção 2) ---
    df = df.merge(datas_df[['atividade_id', 'qt_sessoes', 'qt_horas']], on='atividade_id', how='left')
    df = df.rename(columns={'qt_sessoes': 'sessoes', 'qt_horas': 'horas'})
    df['sessoes'] = pd.to_numeric(df['sessoes'], errors='coerce').fillna(0)
    df['horas']   = pd.to_numeric(df['horas'],   errors='coerce').fillna(0)

    # --- flags de custo INDIVIDUAL (custo desta solicitação, não do total da atividade) ---
    custo_ind = df['custo']
    bool_map  = {1: 'sim', 0: '0'}
    df['acima15mil']  = (custo_ind > 15000).astype(int).map(bool_map)
    df['acima20mil']  = (custo_ind > 20000).astype(int).map(bool_map)
    df['acima100mil'] = (custo_ind > 100000).astype(int).map(bool_map)

    # --- público total estimado da atividade ---
    # tipo_per_capita=0 (Apresentação, Oficina etc.): público repete por sessão → total = sessoes × publico
    # tipo_per_capita=1 (Curso, Seminário): público já é total
    # Acima de 4000: considera total direto (cap para evitar inflação)
    pub = df['publico_sessao'].astype(float)
    tpc = df['tipo_per_capita']
    df['publico'] = np.where(
        pub > 4000, pub,
        np.where(tpc == 0, df['sessoes'] * pub, pub),
    ).astype(int)

    # --- métricas de custo por unidade (baseadas em custo_contratos_total da atividade) ---
    custo_total_ativ = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0)
    df['por_sessao'] = np.where(df['sessoes'] > 0, custo_total_ativ / df['sessoes'], np.nan)
    df['por_hora']   = np.where(df['horas']   > 0, custo_total_ativ / df['horas'],   np.nan)
    df['per_capita'] = np.where(
        tpc == 0,
        np.where(df['publico'] > 0, custo_total_ativ / df['publico'], np.nan),
        np.where(pub > 0, custo_total_ativ / pub, np.nan),
    )

    # --- servico e subatividade para classificar autonomiaCusto ---
    df = df.merge(
        rps_parcial_df[['atividade_id', 'servico', 'subatividade']],
        on='atividade_id', how='left',
    )
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # --- autonomiaCusto baseada no custo INDIVIDUAL desta solicitação ---
    # build_autonomias resolve a hierarquia final tomando drop_duplicates; aqui classificamos por linha
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'

    df['autonomiaCusto'] = np.select(
        [acima100,
         acima20 &  limiar_20k,
         acima20 & ~limiar_20k,
         acima15 & ~limiar_20k],
        ['DIREG', 'STS', 'STS-20', 'STS-15'],
        default='UO',
    )

    # por_hora válido apenas para serviços faturados por horas de execução
    usa_por_hora = (
        df['servico'].isin(SERVICOS_POR_HORA) |
        df['subatividade'].isin(SUBATIV_POR_HORA)
    )
    df['por_hora_valido'] = np.where(usa_por_hora, df['por_hora'], np.nan)

    return df.drop_duplicates(subset=['solicitacao_id'])

In [43]:
# contracts_df: tabela de solicitações administrativas com métricas de custo.
# Executado aqui porque build_contracts() depende de:
#   - raw_solicitacoes_df  (disponível após a seção 8)
#   - rps_parcial_df com tem_passagem já adicionado (rps-passagens-exec acima)
contracts_df = build_contracts(
    raw_solicitacoes_df = raw_solicitacoes_df,
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    rps_parcial_df      = rps_parcial_df,
)
print(f'contracts_df:     {contracts_df.shape}')
print(f'atividades unicas: {contracts_df["atividade_id"].nunique()}')
print()
print('autonomiaCusto:')
print(contracts_df.drop_duplicates("atividade_id")["autonomiaCusto"].value_counts())

contracts_df:     (18465, 30)
atividades unicas: 14215

autonomiaCusto:
autonomiaCusto
UO        11648
STS-20     1495
STS-15      463
STS         398
DIREG       211
Name: count, dtype: int64


## 7. Autonomias

Combina as tabelas de datas, contratos e RPS Parcial para calcular a `autonomia` final.

> **Executa após a seção 8** — depende de `tem_passagem` em `rps_parcial_df`, que é enriquecido via `raw_solicitacoes_df`.

**Fontes e o que contribuem:**
| Fonte | Campo | Regra |
|---|---|---|
| Datas | `autonomiaTemporal` | DIREG (90d/60h), STS (30d corridos), UO — ajustado por serviço |
| Contratos | `autonomiaCusto` | DIREG (>100k), STS-20/STS (20k–100k), STS-15 (15k–20k), UO |
| RPS Parcial | `tag` | STS se `'Avaliação STS'` |
| RPS Parcial | `tem_passagem` | STS se `'Sim'` |

**Hierarquia:** `DIREG > STS > UO` — prevalece sempre o maior nível entre todas as fontes.

`autonomiaCusto` preserva o qualificador (`STS-15`, `STS-20`) para rastreabilidade, mas é tratado como nível STS na hierarquia.

In [44]:
# Hierarquia: DIREG (3) > STS (2) > UO (1)
# Cada fonte de autonomia é convertida para nível numérico;
# o nível máximo determina a autonomia final.

NIVEL = {'DIREG': 3, 'STS': 2, 'STS-20': 2, 'STS-15': 2, 'UO': 1}

def autonomia_nivel(serie: pd.Series) -> pd.Series:
    """Converte valores de autonomia para nível numérico (desconhecido → 1)."""
    return serie.map(NIVEL).fillna(1).astype(int)

NIVEL_INV = {3: 'DIREG', 2: 'STS', 1: 'UO'}

def build_autonomias(rps_df, datas_df, contracts_df) -> pd.DataFrame:
    # Base: uma linha por atividade com serviço cadastrado
    df = rps_df[['atividade_id', 'servico', 'subatividade', 'tag', 'tem_passagem']].copy()

    # --- autonomia temporal (já ajustada por serviço em Datas) ---
    df = df.merge(
        datas_df[['atividade_id', 'autonomiaTemporal']],
        on='atividade_id', how='left'
    )
    df['autonomiaTemporal'] = df['autonomiaTemporal'].fillna('UO')

    # --- autonomia de custo (uma linha por atividade) ---
    custo_por_ativ = (
        contracts_df[['atividade_id', 'autonomiaCusto']]
        .drop_duplicates(subset=['atividade_id'])
    )
    df = df.merge(custo_por_ativ, on='atividade_id', how='left')
    df['autonomiaCusto'] = df['autonomiaCusto'].fillna('UO')

    # --- níveis por fonte ---
    nivel_temporal  = autonomia_nivel(df['autonomiaTemporal'])
    nivel_custo     = autonomia_nivel(df['autonomiaCusto'])
    nivel_tag       = np.where(df['tag'] == 'Avaliação STS', 2, 1)
    nivel_passagem  = np.where(df['tem_passagem'] == 'Sim',  2, 1)

    # --- autonomia final: prevalece o maior nível (DIREG > STS > UO) ---
    nivel_final = pd.concat(
        [nivel_temporal, nivel_custo,
         pd.Series(nivel_tag, index=df.index),
         pd.Series(nivel_passagem, index=df.index)],
        axis=1
    ).max(axis=1)

    df['autonomia'] = nivel_final.map(NIVEL_INV)

    return df


autonomias_df = build_autonomias(rps_parcial_df, datas_df, contracts_df)
autonomias_df['autonomia'].value_counts()

autonomia
UO       27501
STS       3794
DIREG     1536
Name: count, dtype: int64

## 8a. Descrição de custos por atividade (solicitacoes_desc_df)

Uma linha por `atividade_id`; campo `item_desc` com todas as solicitações concatenadas em texto.

**`item_de_custo` → `item_norm`** (normalização):
- Se `item_grupo` for nulo ou `'null'`: usa `nome_item` até o primeiro `'-'` (com trim)
- Mapeamento via `MAPA_ITEM_CUSTO` (match exato) + `PREFIXOS_ITEM_CUSTO` (fallback por prefixo)
- `item_grupo = 'acessibilidade'` → `Acessibilidade` (independente do nome)

**Descartados:** `Camarim` (tipos 1/2), `Água`, `Verificar`, `Estagiário`

**Categorias finais:** `Contrato PJ` · `Contrato PF` · `Contrato Cooperativa` · `Hospedagem` · `Passagem Aérea` · `Turismo` · `Transporte` · `Exibição de Filmes` · `Ação esportiva e recreativa` · `Contratações diversas` · `Compras` · `Alimentação` · `Sonorização` · `Iluminação` · `Audiovisual` · `Locação` · `Comunicação` · `Acessibilidade` · `Outros`

**Ordem em `item_desc`:**
1. Contratos (PJ / PF / Cooperativa)
2. `────────────────────────────`
3. Passagem Aérea
4. `────────────────────────────`
5. Hospedagem
6. `────────────────────────────`
7. Demais categorias — ordem alfabética por `item_norm`, depois `solicitacao_id`

**Saída:** `atividade_id`, `item_desc`, `contratos_desc`, `passagem_desc`, `hospedagem_desc`, `tem_contrato`, `tem_passagem`, `tem_hospedagem`

---

In [45]:
# Itens a descartar antes de construir item_desc
ITENS_DESCARTAR = frozenset([
    'Camarim', 'Camarim Tipo 1', 'Camarim Tipo 2',
    'Água', 'Verificar',
])

# Mapeamento exato: item_de_custo → categoria normalizada
MAPA_ITEM_CUSTO = {
    # Contratos
    'Contrato PJ [Custo cachê/pró-labore]':            'Contrato PJ',
    'Contrato PJ':                                      'Contrato PJ',
    'Contrato PF [Custo cachê/pró-labore]':            'Contrato PF',
    'Contrato PF':                                      'Contrato PF',
    'Contrato Cooperativa [Custo cachê/pró-labore]':   'Contrato Cooperativa',
    # Viagem
    'Hospedagem [Custo hospedagem]':                   'Hospedagem',
    'Passagem Aérea [Custo passagem]':                 'Passagem Aérea',
    'Turismo [Outros custos de terceiros]':            'Turismo',
    'Turismo':                                          'Turismo',
    'Transporte [Outros custos de terceiros]':         'Transporte',
    'Transporte':                                       'Transporte',
    # Serviços artísticos/técnicos
    'Exibição de Filmes':                              'Exibição de Filmes',
    'Ação esportiva e recreativa':                     'Ação esportiva e recreativa',
    'Contratações diversas [Outros custos de terceiros]': 'Contratações diversas',
    # Compras
    'Compras  [Outros custos internos]':               'Compras',
    'Compras':                                          'Compras',
    'Aquisição de material':                           'Compras',
    # Alimentação
    'Kit Lanche':                                       'Alimentação',
    'Brindes':                                          'Alimentação',
    'Café para integrações/Ações Similares':           'Alimentação',
    'Coffee break Tipo 1':                             'Alimentação',
    'Coffee break Tipo 2':                             'Alimentação',
    'Coquetel':                                         'Alimentação',
    'Recepções / Refeições':                           'Alimentação',
    'Diversos - Alimentação [Outros custos de terceiros]': 'Alimentação',
    # Infraestrutura de evento
    'Sonorização':                                      'Sonorização',
    'Locação - Sonorização [Outros custos de terceiros]': 'Sonorização',
    'Iluminação':                                       'Iluminação',
    'Locação - Iluminação [Outros custos de terceiros]': 'Iluminação',
    'Audiovisual / Projeção':                          'Audiovisual',
    'Audiovisual':                                      'Audiovisual',
    'Locação - Outros [Outros custos de terceiros]':   'Locação',
    'Locação':                                          'Locação',
    # Comunicação
    'Impressos e digitais':                            'Comunicação',
    'Editoria web':                                     'Comunicação',
    'Assessoria de imprensa':                          'Comunicação',
    'Diversos - Comunicação [Outros custos de terceiros]': 'Comunicação',
    # Acessibilidade
    'Tradução Simultânea':                             'Acessibilidade',
    # Outros (categorias absorvidas)
    'Serviço de Receptivo':                            'Outros',
    'Mobiliário':                                       'Outros',
    'Montagem':                                         'Outros',
    'Equipamentos':                                     'Outros',
    'Limpeza':                                          'Outros',
    'Elétrica':                                         'Outros',
    'Acompanhamento':                                   'Outros',
    'Outros':                                           'Outros',
    'Outros - Terceiros [Outros custos terceiros]':    'Outros',
    'Outros - Internos [Outros custos internos]':      'Outros',
}

# Fallback por prefixo — para itens com código de conta no nome (ex: 'Hospedagem [3920')
PREFIXOS_ITEM_CUSTO = [
    ('Hospedagem',           'Hospedagem'),
    ('Guia de turismo',      'Turismo'),
    ('Locação de Automóvel', 'Transporte'),
    ('Serviço de Montagem',  'Outros'),
    ('Serviços Gerais',      'Outros'),
]


def normalizar_item_custo(idc: str, item_grupo: str) -> str:
    """Retorna categoria normalizada; None = descartar."""
    idc_s   = str(idc).strip()
    grupo_l = str(item_grupo).strip().lower()
    # Estagiário → descarte
    if 'estagi' in grupo_l or 'estagi' in idc_s.lower():
        return None
    # item_grupo = 'acessibilidade' → Acessibilidade
    if grupo_l == 'acessibilidade':
        return 'Acessibilidade'
    # Match exato
    if idc_s in MAPA_ITEM_CUSTO:
        return MAPA_ITEM_CUSTO[idc_s]
    # Fallback por prefixo
    for prefixo, categoria in PREFIXOS_ITEM_CUSTO:
        if idc_s.startswith(prefixo):
            return categoria
    # Sem mapeamento — mantém o valor bruto
    return idc_s


In [46]:
# Ordem de exibição dos grupos em item_desc
# 0 = Contratos  1 = Passagem Aérea  2 = Hospedagem  3 = demais (alfabético)
GRUPO_ITEM_DESC = {
    'Contrato PJ':          0,
    'Contrato PF':          0,
    'Contrato Cooperativa': 0,
    'Passagem Aérea':       1,
    'Hospedagem':           2,
}
SEP_ITEM_DESC = chr(9472) * 28   # ────────────────────────────


def build_solicitacoes_desc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── item_de_custo bruto ───────────────────────────────────────────────
    grupo = df['item_grupo'].fillna('').astype(str).str.strip()
    nome  = df['nome_item'].fillna('').astype(str).str.strip()
    nome_ate_traco = nome.str.split('-').str[0].str.strip()
    usar_nome = (grupo == '') | (grupo.str.lower() == 'null')
    df['item_de_custo'] = np.where(usar_nome, nome_ate_traco, grupo)

    # ── descarte ─────────────────────────────────────────────────────────
    df = df[~df['item_de_custo'].isin(ITENS_DESCARTAR)].copy()

    # ── normalização ─────────────────────────────────────────────────────
    df['item_norm'] = df.apply(
        lambda r: normalizar_item_custo(r['item_de_custo'], r['item_grupo']),
        axis=1,
    )
    df = df[df['item_norm'].notna()].copy()

    # ── chave de ordenação: (grupo, item_norm alfabético, solicitacao_id) ─
    df['_grp'] = df['item_norm'].map(lambda x: GRUPO_ITEM_DESC.get(x, 3))

    # ── custo formatado ───────────────────────────────────────────────────
    df['custo_fmt'] = df['custo'].apply(
        lambda x: 'R$ ' + f'{x:,.0f}'.replace(',', '.')
    )

    # ── linha individual ──────────────────────────────────────────────────
    desc = df['descricao'].fillna('').astype(str).str.strip()
    df['linha'] = df['item_norm'] + ' — ' + desc + ' — ' + df['custo_fmt']

    # ── agrega por atividade_id com separadores entre grupos ─────────────
    def montar_desc(sub):
        sub = sub.sort_values(['_grp', 'item_norm', 'solicitacao_id'])
        linhas = []
        grp_atual = None
        for _, row in sub.iterrows():
            if grp_atual is not None and row['_grp'] != grp_atual:
                linhas.append(SEP_ITEM_DESC)
            linhas.append(row['linha'])
            grp_atual = row['_grp']
        return chr(10).join(linhas)

    def agrupar(sub_df, col_name: str) -> pd.DataFrame:
        if sub_df.empty:
            return pd.DataFrame(columns=['atividade_id', col_name])
        return (
            sub_df.groupby('atividade_id')
            .apply(montar_desc)
            .reset_index(name=col_name)
        )

    # ── item_desc completo ────────────────────────────────────────────────
    resultado = (
        df.groupby('atividade_id')
        .apply(montar_desc)
        .reset_index(name='item_desc')
    )

    # ── descrições por categoria ──────────────────────────────────────────
    resultado = (
        resultado
        .merge(agrupar(df[df['_grp'] == 0],                    'contratos_desc'), on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Passagem Aérea'], 'passagem_desc'),  on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Hospedagem'],     'hospedagem_desc'), on='atividade_id', how='left')
    )

    # ── flags 0/1 ─────────────────────────────────────────────────────────
    resultado['tem_contrato']   = resultado['contratos_desc'].notna().astype(int)
    resultado['tem_passagem']   = resultado['passagem_desc'].notna().astype(int)
    resultado['tem_hospedagem'] = resultado['hospedagem_desc'].notna().astype(int)

    for col in ['contratos_desc', 'passagem_desc', 'hospedagem_desc']:
        resultado[col] = resultado[col].fillna('')

    return resultado


solicitacoes_desc_df = build_solicitacoes_desc(raw_solicitacoes_df)

print(f'solicitacoes_desc_df: {solicitacoes_desc_df.shape}')
print(f'  tem_contrato=1:   {solicitacoes_desc_df["tem_contrato"].sum()}')
print(f'  tem_passagem=1:   {solicitacoes_desc_df["tem_passagem"].sum()}')
print(f'  tem_hospedagem=1: {solicitacoes_desc_df["tem_hospedagem"].sum()}')
print()
print('Exemplo (primeira linha de item_desc):')
print(solicitacoes_desc_df['item_desc'].iloc[0])

solicitacoes_desc_df: (16056, 8)
  tem_contrato=1:   13792
  tem_passagem=1:   530
  tem_hospedagem=1: 2684

Exemplo (primeira linha de item_desc):
Contrato PJ — Luiza Helena Craemer Franscesconi - ME — R$ 8.775


## 9. PCAP por solicitação (raw_pcap_df)

Vincula solicitações do Siplan às propostas PCAP correspondentes.

**Reutiliza `raw_solicitacoes_df`** para o lado siplan_solicitacao — sem segunda query ao Hive.
A única query nova é para `stg_pcap.vw_listagem_proposta`.

**Lógica:**
1. Filtra `raw_solicitacoes_df`: `area = 'Administrativo'` + `descricao` com padrão `PCAP[^0-9]*[0-9]{13}`
2. Extrai `pcap_num` (13 dígitos) via regex Python
3. SQL: extrai `pcap_num` de `proposta.proposta` na view PCAP
4. Inner join em `pcap_num`
5. `drop_duplicates(subset=['solicitacao_id'])` — equivalente ao "Duplicatas Removidas" do Power Query

**Campos:**
- `atividade_id`, `solicitacao_id`, `proposta` (pcap_num) — da solicitação
- `pcap_status`, `pcap_nome`, `pcap_razao_social` — da proposta
- `pcap_cache`, `pcap_producao`, `pcap_admin`, `pcap_tributo`, `pcap_demais_itens`, `pcap_total` — valores

**Flag na tabela base:** `com_pcap` = 1 se `atividade_id` presente em `raw_pcap_df`

> Lakehouse Staging: **`raw_pcap`**

In [47]:
# Lakehouse Staging: raw_pcap (lado vw_listagem_proposta)
sql_raw_pcap = f'''
SELECT
    CAST(REGEXP_EXTRACT(proposta, '(?i)PCAP[^0-9]*([0-9]{{13}})', 1) AS BIGINT) AS pcap_num,
    status                AS pcap_status,
    nome_atividade        AS pcap_nome,
    proponente_rz         AS pcap_razao_social,
    valor_cache           AS pcap_cache,
    valor_de_producao     AS pcap_producao,
    valor_admin           AS pcap_admin,
    valor_tributo_c       AS pcap_tributo,
    valor_demais_itens    AS pcap_demais_itens,
    valor_total_abc       AS pcap_total
FROM stg_pcap.vw_listagem_proposta
WHERE YEAR(data_atividade_1) = YEAR(DATE '{DATA_INICIAL}')
'''

In [ ]:
import re

# ── lado siplan_solicitacao: reutiliza raw_solicitacoes_df ────────────────
pcap_re = r'PCAP[^0-9]*([0-9]{13})'

sol_pcap = raw_solicitacoes_df[
    (raw_solicitacoes_df['area'] == 'Administrativo') &
    raw_solicitacoes_df['descricao'].str.contains(pcap_re, flags=re.IGNORECASE, na=False)
][['atividade_id', 'solicitacao_id', 'descricao']].copy()

# Extrai o número de 13 dígitos e salva como pcap_num (nome usado no merge abaixo)
sol_pcap['pcap_num'] = (
    sol_pcap['descricao']
    .str.extract(pcap_re, flags=re.IGNORECASE)[0]
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
sol_pcap = sol_pcap.drop(columns=['descricao']).drop_duplicates(subset=['solicitacao_id'])

# ── lado vw_listagem_proposta: query nova ─────────────────────────────────
pcap_props_df = query_to_df(sql_raw_pcap)
pcap_props_df.columns = [col.split('.')[-1] for col in pcap_props_df.columns]
pcap_props_df['pcap_num'] = pd.to_numeric(pcap_props_df['pcap_num'], errors='coerce').astype('Int64')

# ── join ──────────────────────────────────────────────────────────────────
raw_pcap_df = sol_pcap.merge(pcap_props_df, on='pcap_num', how='inner')

print(f'raw_pcap_df:        {raw_pcap_df.shape}')
print(f'atividades únicas:  {raw_pcap_df["atividade_id"].nunique()}')
print(f'solicitações únicas:{raw_pcap_df["solicitacao_id"].nunique()}')

## 10. Tabela Base (tabela_base_df)

Join final de todas as tabelas staging. **Uma linha por `atividade_id`.**

**Fontes e contribuições:**

| Fonte | O que traz | Estratégia |
|---|---|---|
| `raw_acoes_df` | base — identificação, hierarquia, classificação, público | `drop('projeto')` — substituído por `raw_projetos_df.projeto_nome` |
| `datas_df` | datas, sessões, horas, flags de prazo e ano | LEFT JOIN |
| `autonomias_df` | `autonomia`, `autonomiaTemporal`, `autonomiaCusto` | LEFT JOIN — apenas estas 3 colunas |
| `raw_projetos_df` | metadados de projeto, `tem_pai` | LEFT JOIN — `drop('institucional')` |
| `todas_as_datas_df` | `todas_as_datas` (texto) | LEFT JOIN |
| `raw_tags_df` | `todas_as_tags` | LEFT JOIN após `drop_duplicates('atividade_id')` |
| `solicitacoes_desc_df` | `item_desc`, `*_desc` por categoria, `tem_contrato/passagem/hospedagem` | LEFT JOIN |
| `contracts_df` | `custo_contratos_total`, `custo_total`, `n_contratos`, `n_solic` | LEFT JOIN após `drop_duplicates('atividade_id')` |
| `raw_acessibilidade_df` | — | `.isin()` → flag `tem_dispositivo` (0/1) |
| `raw_pcap_df` | — | `.isin()` → flag `com_pcap` (0/1) |

**Campos derivados calculados aqui:**

| Campo | Lógica |
|---|---|
| `faixa` | `np.select` duplo sobre `idade_inicial/final`, `linguagem`, `recomendacao_etaria` → `'infantil'`/`'pessoas idosas'`/`'s/i'` |
| `periodicidade` | Lista permanente / eventual + regra Curso/Vivência (dias > 90 e sessões > 30) → `'permanente'`/`'eventual'`/`'s/i'` |
| `tem_dispositivo` | `atividade_id ∈ raw_acessibilidade_df` → 0/1 |
| `com_pcap` | `atividade_id ∈ raw_pcap_df` → 0/1 |
| `espaco_brincar` | OR de 4 fontes (nome, tag, projeto_uo_nome, local/tipologia) → 0/1 |

> Lakehouse Prod: **`tabela_base`**

In [ ]:
SUBATIV_PERMANENTE = frozenset({
    'Acesso a recursos informacionais',
    'Análise de risco em saúde',
    'Consulta',
    'Exercício físicos sistematicos',
    'Formação esportiva',
    'Sessão diagnóstica/clínica',
    'Refeição',
    'Práticas coletivas',
    'Parque aquático',
    'Multipráticas recreativas',
    'Lanche',
    'Hospedagem',
})

SUBATIV_EVENTUAL = frozenset({
    'Apresentação',
    'Competições físico-esportivas',
    'Eventos',
    'Exibição',
    'Exposição',
    'Passeios',
    'Produtos gastronômicos',
    'Viagens',
})


def build_tabela_base(
    raw_acoes_df, datas_df, autonomias_df, raw_projetos_df,
    todas_as_datas_df, raw_tags_df, solicitacoes_desc_df, contracts_df,
    raw_acessibilidade_df, raw_pcap_df, raw_datas_sessoes_df,
    precif_df,
) -> pd.DataFrame:

    df = raw_acoes_df.drop(columns=['projeto'], errors='ignore').copy()

    # ── faixa etária ──────────────────────────────────────────────────────
    df['idade_inicial'] = pd.to_numeric(df['idade_inicial'], errors='coerce').fillna(0)
    df['idade_final']   = pd.to_numeric(df['idade_final'],   errors='coerce').fillna(0)
    temp0 = df['idade_inicial'] + df['idade_final']
    temp1 = np.select(
        [df['linguagem'] == 'Crianças',
         df['linguagem'] == 'Idosos',
         df['recomendacao_etaria'] != 'Livre'],
        ['infantil', 'pessoas idosas', 'não é'],
        default='pode ser',
    )
    temp2 = np.select(
        [temp0 == 0, df['idade_inicial'] >= 60, df['idade_inicial'] >= 12,
         df['idade_final'] == 0, df['idade_final'] <= 13, df['idade_final'] <= 15],
        ['s/i', 'idosos', 'não é', 's/i', 'infantil', 'pode ser'],
        default='não é',
    )
    mesclado = pd.Series(np.array(temp1, dtype=object) + ' - ' + np.array(temp2, dtype=object), index=df.index)
    df['faixa'] = np.select(
        [mesclado.str.contains('infantil'), mesclado.str.contains('idosos'),
         mesclado.str.contains('pode ser'), mesclado.str.contains('s/i')],
        ['infantil', 'pessoas idosas', 's/i', 's/i'],
        default='s/i',
    )

    # ── joins 1:1 ─────────────────────────────────────────────────────────
    datas_cols = [
        'atividade_id', 'primeiradata', 'PrimeiraData', 'PrimeiraHora',
        'ultimadata', 'qt_sessoes', 'qt_datas_distintas', 'qt_horas',
        'tempo_da_sessao', 'diascorridos', 'mes',
        '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
    ]
    df = df.merge(
        datas_df[[c for c in datas_cols if c in datas_df.columns]],
        on='atividade_id', how='left',
    )
    df = df.merge(
        autonomias_df[['atividade_id', 'autonomia', 'autonomiaTemporal', 'autonomiaCusto']],
        on='atividade_id', how='left',
    )
    df = df.merge(
        raw_projetos_df.drop(columns=['institucional'], errors='ignore'),
        on='atividade_id', how='left',
    )
    df = df.merge(todas_as_datas_df, on='atividade_id', how='left')
    df = df.merge(
        raw_tags_df.drop_duplicates('atividade_id')[['atividade_id', 'todas_as_tags']],
        on='atividade_id', how='left',
    )
    df = df.merge(solicitacoes_desc_df, on='atividade_id', how='left')
    df = df.merge(
        contracts_df.drop_duplicates('atividade_id')[
            ['atividade_id', 'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic']
        ],
        on='atividade_id', how='left',
    )
    df = df.merge(
        precif_df[['atividade_id', 'gratuito', 'maior_valor', 'menor_valor']],
        on='atividade_id', how='left',
    )

    # ── periodicidade ─────────────────────────────────────────────────────
    # Regras (mutuamente exclusivas):
    # 1. Subatividades sempre permanentes
    # 2. Subatividades sempre eventuais (Ações formativas/mediadas excluem Curso/Vivência)
    # 3. Curso (subativ=Ações formativas) e Vivência (subativ=Ações mediadas):
    #    diascorridos > 90 E qt_sessoes > 30 → permanente; caso contrário → eventual
    dias = pd.to_numeric(df['diascorridos'], errors='coerce').fillna(0)
    sess = pd.to_numeric(df['qt_sessoes'],   errors='coerce').fillna(0)

    cond_perm  = df['subatividade'].isin(SUBATIV_PERMANENTE)
    cond_ev    = (
        df['subatividade'].isin(SUBATIV_EVENTUAL) |
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] != 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] != 'Vivência'))
    )
    cond_cv    = (
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] == 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] == 'Vivência'))
    )
    cv_perm = cond_cv & (dias > 90) & (sess > 30)
    cv_ev   = cond_cv & ~((dias > 90) & (sess > 30))

    df['periodicidade'] = np.select(
        [cond_perm, cond_ev, cv_perm, cv_ev],
        ['permanente', 'eventual', 'permanente', 'eventual'],
        default='s/i',
    )

    # ── flags via .isin() ─────────────────────────────────────────────────
    df['tem_dispositivo'] = df['atividade_id'].isin(raw_acessibilidade_df['atividade_id']).astype(int)
    df['com_pcap']        = df['atividade_id'].isin(raw_pcap_df['atividade_id']).astype(int)

    # ── espaco_brincar: OR de quatro fontes ───────────────────────────────
    _eb = set().union(
        raw_acoes_df.loc[raw_acoes_df['nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_tags_df.loc[raw_tags_df['tag_nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_projetos_df.loc[
            raw_projetos_df['projeto_uo_nome'].str.contains('Espaço de Brincar', na=False) |
            raw_projetos_df['tag_projeto'].str.contains('Espaço de Brincar', na=False),
            'atividade_id',
        ],
        raw_datas_sessoes_df.loc[
            raw_datas_sessoes_df['local'].str.contains('Espaço de Brincar', na=False) |
            raw_datas_sessoes_df['TipologiaLocal'].str.contains('Espaço de Brincar', na=False),
            'atividade_id',
        ],
    )
    df['espaco_brincar'] = df['atividade_id'].isin(_eb).astype(int)

    # ── tem_educador: educadores no campo complemento ─────────────────────────
    # Equivale ao DAX SWITCH(TRUE(), SEARCH("educador",[complemento])>0, TRUE, ...)
    # Cobre: educador, educadora, educadoras, educadores, agente de educação ambiental
    df['tem_educador'] = df['complemento'].fillna('').str.contains(
        r'educador[ae]?s?|agente de educa[cç][aã]o ambiental', case=False
    ).astype(int)

    # ── reordenação de colunas ────────────────────────────────────────────
    col_order = [
        # Identificação
        'uo', 'atividade_id', 'status_atividade', 'nome', 'complemento',
        # Hierarquia programática
        'areaprog', 'atividade', 'subatividade', 'servico', 'periodicidade',
        # Classificação
        'tipo', 'subtipo', 'formato', 'linguagem',
        # Público e faixa etária
        'recomendacao_etaria', 'faixa', 'estimativa_publico', 'lugares',
        # Datas e sessões
        'primeiradata', 'PrimeiraData', 'PrimeiraHora', 'ultimadata',
        'qt_sessoes', 'qt_datas_distintas', 'qt_horas', 'tempo_da_sessao',
        'diascorridos', 'mes', '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        # Flags de ano
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
        # Texto de datas
        'todas_as_datas',
        # Autonomia
        'autonomia', 'autonomiaTemporal', 'autonomiaCusto',
        # Projeto
        'projeto_id', 'projeto_nome', 'projeto_complemento', 'projeto_categoria',
        'tag_projeto', 'tag_grupo_projeto', 'projeto_uo_nome',
        'projeto_descricao', 'projeto_comunicacao', 'projeto_conceitual', 'tem_pai',
        # Tags
        'todas_as_tags',
        # Custos — flags e totais
        'tem_contrato', 'tem_passagem', 'tem_hospedagem',
        'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic',
        # Custos — descrições
        'item_desc', 'contratos_desc', 'passagem_desc', 'hospedagem_desc',
        # Flags especiais
        'tem_dispositivo', 'espaco_brincar', 'com_pcap', 'tem_educador',
        # Complementares
        'gratuito', 'maior_valor', 'menor_valor', 'precificacao_desc', 'produtor', 'tem_parceria',
        'contatofornecedores', 'uso_interno', 'manutencao', 'regular',
        'integracao_sgc', 'institucional',
    ]
    ordered = [c for c in col_order if c in df.columns]
    extras  = [c for c in df.columns if c not in ordered]
    return df[ordered + extras]

In [ ]:
tabela_base_df = build_tabela_base(
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    autonomias_df       = autonomias_df,
    raw_projetos_df     = raw_projetos_df,
    todas_as_datas_df   = todas_as_datas_df,
    raw_tags_df         = raw_tags_df,
    solicitacoes_desc_df= solicitacoes_desc_df,
    contracts_df        = contracts_df,
    raw_acessibilidade_df = raw_acessibilidade_df,
    raw_pcap_df         = raw_pcap_df,
    raw_datas_sessoes_df= raw_datas_sessoes_df,
    precif_df           = precif_df,
)

print(f'tabela_base_df: {tabela_base_df.shape}')
print(f'atividade_id única: {tabela_base_df["atividade_id"].is_unique}')
print()
print('faixa:')
print(tabela_base_df['faixa'].value_counts())
print()
print('autonomia:')
print(tabela_base_df['autonomia'].value_counts())
print()
flags = ['tem_dispositivo', 'espaco_brincar', 'com_pcap',
         'tem_contrato', 'tem_passagem', 'tem_hospedagem']
for f in flags:
    if f in tabela_base_df.columns:
        print(f'{f}=1: {tabela_base_df[f].sum()}')